In [21]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.naive_bayes import MultinomialNB, GaussianNB, BernoulliNB
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
import joblib

In [3]:
df=pd.read_csv('https://raw.githubusercontent.com/4GeeksAcademy/naive-bayes-project-tutorial/main/playstore_reviews.csv')
df

,package_name,review,polarity
0,com.facebook.katana,privacy at least put some option appear offli...,0
1,com.facebook.katana,"messenger issues ever since the last update, ...",0
2,com.facebook.katana,profile any time my wife or anybody has more ...,0
3,com.facebook.katana,the new features suck for those of us who don...,0
4,com.facebook.katana,forced reload on uploading pic on replying co...,0
...,...,...,...
886,com.rovio.angrybirds,loved it i loooooooooooooovvved it because it...,1
887,com.rovio.angrybirds,all time legendary game the birthday party le...,1
888,com.rovio.angrybirds,ads are way to heavy listen to the bad review...,0
889,com.rovio.angrybirds,fun works perfectly well. ads aren't as annoy...,1


In [4]:
# Elimina la columna 'package_name'
df.drop('package_name', axis=1, inplace=True)
df

,review,polarity
0,privacy at least put some option appear offli...,0
1,"messenger issues ever since the last update, ...",0
2,profile any time my wife or anybody has more ...,0
3,the new features suck for those of us who don...,0
4,forced reload on uploading pic on replying co...,0
...,...,...
886,loved it i loooooooooooooovvved it because it...,1
887,all time legendary game the birthday party le...,1
888,ads are way to heavy listen to the bad review...,0
889,fun works perfectly well. ads aren't as annoy...,1


# Eliminar espacios y convertir a minúsculas el texto

In [5]:
df ['review']= df['review'].str.strip().str.lower()

# Dividiemos en variables el DF

In [6]:
review = df['review']
polarity = df['polarity']

# Dividir el conjunto de datos en train y test

In [7]:
X_train, X_test, y_train, y_test = train_test_split(review, polarity, test_size=0.2, random_state=42)

# Transformar el texto en una matriz de recuento de palabras

In [8]:
vectorizer = CountVectorizer()

X_train_vec = vectorizer.fit_transform(X_train).toarray()
X_test_vec = vectorizer.transform(X_test).toarray()

# MultinomialNB 

In [9]:
bayes = MultinomialNB().fit(X_train_vec, y_train)
y_pred = bayes.predict(X_test_vec)
print(classification_report(y_test, y_pred))


              precision    recall  f1-score   support

           0       0.85      0.95      0.90       126
           1       0.84      0.58      0.69        53

    accuracy                           0.84       179
   macro avg       0.84      0.77      0.79       179
weighted avg       0.84      0.84      0.83       179



# Modelo GaussianNB

In [10]:
bayes = GaussianNB().fit(X_train_vec, y_train)
y_pred = bayes.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.84      0.89      0.86       126
           1       0.70      0.60      0.65        53

    accuracy                           0.80       179
   macro avg       0.77      0.75      0.76       179
weighted avg       0.80      0.80      0.80       179



# Modelo BernoulliNB

In [24]:
bayes_default = BernoulliNB().fit(X_train_vec, y_train)
y_pred = bayes.predict(X_test_vec)
print(classification_report(y_test, y_pred))

              precision    recall  f1-score   support

           0       0.86      0.94      0.90       126
           1       0.81      0.64      0.72        53

    accuracy                           0.85       179
   macro avg       0.84      0.79      0.81       179
weighted avg       0.85      0.85      0.84       179



al implementar los tres modelos, vemos que el que nos arroja mejor resultado (acurracy) es el BernoulliNB por lo cual escogemos este para opbimizar

# Optimizar Hiperparametros

In [12]:

param_grid = {
    'alpha': [0.01, 0.1, 0.5, 1, 2],
    'binarize': [None, 0.0, 0.5, 1.0],
    'fit_prior': [True, False]
}

model = BernoulliNB()
grid = GridSearchCV(model, param_grid, cv=5, scoring='f1_macro')
grid.fit(X_train_vec, y_train)

print("Mejores parámetros:", grid.best_params_)


/home/vscode/.local/lib/python3.11/site-packages/sklearn/naive_bayes.py:1224: RuntimeWarning: invalid value encountered in log
  neg_prob = np.log(1 - np.exp(self.feature_log_prob_))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/naive_bayes.py:1224: RuntimeWarning: invalid value encountered in log
  neg_prob = np.log(1 - np.exp(self.feature_log_prob_))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/naive_bayes.py:1224: RuntimeWarning: invalid value encountered in log
  neg_prob = np.log(1 - np.exp(self.feature_log_prob_))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/naive_bayes.py:1224: RuntimeWarning: invalid value encountered in log
  neg_prob = np.log(1 - np.exp(self.feature_log_prob_))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/naive_bayes.py:1224: RuntimeWarning: invalid value encountered in log
  neg_prob = np.log(1 - np.exp(self.feature_log_prob_))
/home/vscode/.local/lib/python3.11/site-packages/sklearn/naive_bayes.py:1224: Runtime

Mejores parámetros: {'alpha': 0.01, 'binarize': 0.0, 'fit_prior': False}


In [14]:
y_pred = grid.predict(X_test_vec)
print("Accuracy:", accuracy_score(y_test, y_pred))

Accuracy: 0.8491620111731844


Notamos que nos da un mejor acurracy el modelo default que al que le buscamos los hiperparametros

## Random forest

In [16]:
model_rf = RandomForestClassifier(random_state=42)
model_rf.fit(X_train_vec, y_train)
y_pred_rf = model_rf.predict(X_test_vec)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.8268156424581006


### Optimizamos

In [23]:
param_grid_rf = {
    'n_estimators': [100, 300],
    'max_depth': [None, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2],
}
model_rf = RandomForestClassifier(random_state=42)
grid_rf = GridSearchCV(model_rf, param_grid_rf, cv=5, scoring='f1_macro')
grid_rf.fit(X_train_vec, y_train)
grid_rf.best_estimator_
best_rf = grid_rf.best_estimator_
print("Mejores parámetros Random Forest:", grid_rf.best_params_)

Mejores parámetros Random Forest: {'max_depth': None, 'min_samples_leaf': 1, 'min_samples_split': 5, 'n_estimators': 300}


In [20]:
y_pred_rf = grid_rf.predict(X_test_vec)
print("Random Forest Grid Search Accuracy:", accuracy_score(y_test, y_pred_rf))

Random Forest Grid Search Accuracy: 0.8435754189944135


Vemos que el BernoulliNB default sigue siendo la mejor opcion.

# Guardamos los modelos

In [25]:
joblib.dump(bayes_default, 'naive_bayes_model.joblib')
joblib.dump(best_rf, 'random_forest_model.joblib')

['random_forest_model.joblib']